<!-- notebook-header -->
# Probabilidade II: Topicos Avancados para ML

**Modulo:** 00 - Matematica  
**Tipo:** Aula com exercicios guiados e solucoes executaveis  
**Descricao:** Distribuicoes conjuntas, MLE, inferencia bayesiana, normal multivariada e GMM.


# Probabilidade II: Topicos Avancados para ML

**Tempo estimado:** 10-12 horas
**Pre-requisitos:** 0.3 (Matrizes), 0.6 (Probabilidade I)
**Proximo modulo:** 0.8 (Otimizacao para ML)

---

## Indice

1. Introducao: De probabilidade basica a modelos generativos
2. Distribuicoes conjuntas e marginais
3. Covariancia e correlacao de Pearson
4. Correlacao vs Causalidade (confundidores)
5. Distribuicoes condicionais e slicing
6. Lei dos Grandes Numeros (demonstracao)
7. Teorema Central do Limite
8. Estimacao de parametros e propriedades
9. Maxima Verossimilhanca (MLE)
10. Conexao MLE e Cross-Entropy Loss
11. Inferencia Bayesiana
12. Regularizacao como Prior Gaussiano
13. Distribuicao Normal Multivariada
14. Mistura de Gaussianas (GMM)
15. Exercicios integradores
16. Erros Comuns
17. Resumo e Conexoes

## Pre-requisitos e Fio Narrativo

| Topico | Notebook | Por que precisa |
|--------|----------|-----------------|
| Matrizes e operacoes | 0.3 | Covariancia e uma matriz; Normal multivariada usa inversao |
| Derivadas e gradientes | 0.4 | MLE resolve dL/d(theta) = 0; otimizacao de likelihood |
| Integrais | 0.5 | Marginalizacao = integrar sobre variaveis |
| Probabilidade I | 0.6 | Bayes, distribuicoes univariadas, esperanca/variancia |

**Fio narrativo deste notebook:**

O notebook 0.6 construiu os fundamentos: axiomas, distribuicoes univariadas, Bayes. Agora damos o salto para o mundo real do ML, onde:

1. **Multiplas variaveis** interagem (secoes 2-5): dados reais tem muitas features
2. **Convergencia e estimacao** (secoes 6-9): como extrair parametros dos dados
3. **Conexao com ML** (secoes 10-12): por que Cross-Entropy, por que regularizacao
4. **Modelos generativos** (secoes 13-14): a fronteira com deep learning

Ao final, voce entendera por que minimizar Cross-Entropy = maximizar verossimilhanca, e por que regularizacao = Bayesianismo.

## Por que Probabilidade Avancada em ML?

**Analogia:** Se Probabilidade I deu o vocabulario, este notebook ensina a gramatica -- como combinar palavras (variaveis) em frases (modelos) que fazem sentido.

| Conceito | Aplicacao direta em ML | Secao |
|----------|----------------------|-------|
| Distribuicoes conjuntas | Modelar relacoes entre features | 2 |
| Correlacao e causalidade | Selecao de features, analise exploratoria | 3-4 |
| Teorema Central do Limite | Justifica testes estatisticos e mini-batch SGD | 7 |
| MLE | Treinamento de qualquer modelo parametrico | 9 |
| Cross-Entropy = MLE | Por que usamos essa loss em classificacao | 10 |
| Inferencia Bayesiana | Quantificar incerteza nas predicoes | 11 |
| Regularizacao como Prior | Ridge/Lasso tem interpretacao probabilistica | 12 |
| Normal Multivariada | PCA, LDA, modelos generativos | 13 |
| GMM | Clustering probabilistico, deteccao de anomalias | 14 |

## 1. Introducao: De Probabilidade Basica a Modelos Generativos

No modulo 0.6, trabalhamos com uma variavel por vez: P(X), E[X], Var[X].

Agora expandimos em tres direcoes:

**Direcao 1 -- Multiplas variaveis:** Como X e Y se relacionam? A resposta e P(X,Y), a distribuicao conjunta. Dela extraimos correlacao, condicional, e independencia.

**Direcao 2 -- Estimacao:** Dado dados, como descobrir os parametros do modelo? MLE encontra o theta que torna os dados mais provaveis. Bayes vai alem, fornecendo distribuicoes sobre theta.

**Direcao 3 -- Modelos generativos:** Se conhecemos P(X), podemos gerar novos dados! GMM, VAEs e Diffusion Models exploram essa ideia.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# from scipy import stats, optimize
import seaborn as sns
np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

## 2. Distribuicoes Conjuntas e Marginais

### Intuicao: De Uma para Multiplas Variaveis

No modulo anterior, estudamos P(X) isoladamente. Mas no mundo real, features interagem: quem faz mais exercicio (X) tende a ter menos peso (Y)?

**Distribuicao conjunta P(X,Y):** descreve a probabilidade de PARES (x,y). Contem toda a informacao sobre a relacao entre X e Y.

**Distribuicao marginal P(X):** obtida "esquecendo" Y:
$$P(X) = \int P(X,Y)\, dY$$

**Intuicao da marginalizacao:** Imagine uma tabela 2D com probabilidades. Somar uma coluna inteira da a probabilidade marginal daquela linha. A marginal "projeta" a conjunta em um eixo.

| Conceito | Formula | Informacao |
|----------|---------|------------|
| Conjunta P(X,Y) | Tabela/superficie 2D | Relacao completa entre X e Y |
| Marginal P(X) | Soma/integral sobre Y | Comportamento de X ignorando Y |
| Marginal P(Y) | Soma/integral sobre X | Comportamento de Y ignorando X |

**Por que em ML:** Quando marginalizamos, perdemos informacao sobre a relacao. Por isso modelos como regressao trabalham com P(Y dado X), nao com P(Y) sozinha.

In [ ]:
np.random.seed(42)
data = np.random.multivariate_normal([0, 0], [[1, 0.6], [0.6, 1]], 1000)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
ax.scatter(data[:, 0], data[:, 1], alpha=0.5, s=20)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title('Distribuicao Conjunta P(X,Y)')
ax.grid(alpha=0.3)
ax.set_aspect('equal')

ax = axes[1]
ax.hist(data[:, 0], bins=40, density=True, alpha=0.7, color='blue', edgecolor='black')
ax.set_xlabel('X')
ax.set_ylabel('P(X)')
ax.set_title('Distribuicao Marginal P(X)')
ax.grid(alpha=0.3, axis='y')

ax = axes[2]
ax.hist(data[:, 1], bins=40, density=True, alpha=0.7, color='red', edgecolor='black')
ax.set_xlabel('Y')
ax.set_ylabel('P(Y)')
ax.set_title('Distribuicao Marginal P(Y)')
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print('Propriedades:')
print('P(X,Y) eh tabela 2D (ou densidade)')
print('P(X) eh margem da tabela (soma coluna)')
print('P(Y) eh margem da tabela (soma linha)')

**O que observar:** O grafico esquerdo mostra a nuvem 2D (conjunta) com correlacao positiva. Os histogramas nas margens mostram as marginais P(X) e P(Y), ambas normais.

**O que concluir:** A conjunta contem TODA a informacao. As marginais sao resumos que "projetam" a nuvem 2D em cada eixo. Note que a correlacao entre X e Y "desaparece" quando olhamos as marginais separadamente -- cada uma parece uma Normal simples.

**Conexao com secao 3:** Para quantificar a relacao que vemos na nuvem 2D, precisamos de covariancia e correlacao.

## 3. Covariancia e Correlacao de Pearson

### Intuicao: Medindo Associacao Linear

**Covariancia** mede como duas variaveis "caminham juntas":
$$\text{Cov}(X,Y) = E[(X-\mu_X)(Y-\mu_Y)]$$

Se quando X esta acima da media, Y tambem tende a estar → Cov > 0.

**Problema:** Cov depende da escala das variaveis. Cov(altura_cm, peso_kg) != Cov(altura_m, peso_g).

**Correlacao de Pearson** normaliza em [-1, 1]:
$$\rho = \frac{\text{Cov}(X,Y)}{\sigma_X \sigma_Y}$$

| rho | Significado | Formato da nuvem |
|-----|-------------|-----------------|
| +1 | Relacao positiva perfeita | Linha reta subindo |
| 0 | Sem relacao LINEAR | Nuvem redonda |
| -1 | Relacao negativa perfeita | Linha reta descendo |

**Armadilha critica:** rho mede apenas relacao LINEAR. Pode haver relacao nao-linear forte com rho ≈ 0 (ex: Y = X^2 tem rho ≈ 0 se X e simetrico em torno de 0).

**Por que em ML:** Correlacao e o primeiro passo da analise exploratoria. Features altamente correlacionadas sao redundantes (multicolinearidade). PCA remove essas correlacoes.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

rho_vals = [-0.95, -0.5, 0.0, 0.5, 0.8]

for idx, rho in enumerate(rho_vals):
    ax = axes[idx // 3, idx % 3]

    cov_mat = [[1, rho], [rho, 1]]
    data_corr = np.random.multivariate_normal([0, 0], cov_mat, 500)

    ax.scatter(data_corr[:, 0], data_corr[:, 1], alpha=0.6, s=40)
    ax.set_xlim([-4, 4])
    ax.set_ylim([-4, 4])
    ax.set_title(f'Correlacao rho = {rho}')
    ax.grid(alpha=0.3)
    ax.set_aspect('equal')

    empirical_rho = np.corrcoef(data_corr[:, 0], data_corr[:, 1])[0, 1]
    ax.text(0.05, 0.95, f'empirico={empirical_rho:.3f}', transform=ax.transAxes,
            fontsize=9, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

axes[1, 2].remove()

plt.suptitle('Correlacao de Pearson: -1 (negativa) a +1 (positiva)')
plt.tight_layout()
plt.show()

**O que observar:** De esquerda para direita, rho vai de -0.95 ate +0.8. A nuvem "gira" de inclinacao negativa para positiva. Quando rho = 0, a nuvem e redonda.

**O que concluir:** Correlacao descreve a "inclinacao" da relacao linear entre X e Y. E uma medida simples e poderosa, mas tem limitacoes: nao captura relacoes nao-lineares, e nao implica causalidade.

**Conexao com secao 4:** A proxima secao mostra por que correlacao != causalidade, talvez o erro mais perigoso em analise de dados.

## 4. Correlacao vs Causalidade (IMPORTANTE!)

### Intuicao: O Erro Mais Comum em Data Science

Observar correlacao entre X e Y pode significar:

| Explicacao | Diagrama | Exemplo |
|------------|----------|---------|
| X causa Y | X → Y | Fumar → Cancer |
| Y causa X | Y → X | Exercicio ← Motivacao |
| Confundidor Z | Z → X, Z → Y | Idade → Redes Sociais, Idade → Depressao |
| Coincidencia | X ... Y | Consumo de sorvete e afogamentos |

**O confundidor** e uma variavel oculta que causa AMBAS as variaveis observadas, criando uma correlacao espuria.

**Exemplo classico:** Estudos mostram correlacao entre uso de redes sociais e depressao. Conclusao apressada: redes sociais causam depressao. Mas a idade e um confundidor: adolescentes usam mais redes sociais E tem mais depressao por razoes biologicas.

**Por que em ML:** Se seu modelo aprende correlacao espuria (confundidor), ele vai falhar quando o confundidor mudar. Isso e chamado *distribution shift* e e um dos maiores problemas praticos.

In [ ]:
np.random.seed(42)
n = 500

# Cenario: confundidor (idade)
age = np.random.normal(25, 10, n)
age = np.clip(age, 10, 60)

# X e Y causados por idade (nao um pelo outro!)
social_media_hours = 5 + 0.3 * age + np.random.normal(0, 15, n)
depression_score = 20 + 0.5 * age + np.random.normal(0, 10, n)

rho_observed = np.corrcoef(social_media_hours, depression_score)[0, 1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.scatter(social_media_hours, depression_score, c=age, cmap='viridis',
           alpha=0.6, s=50)
ax1.set_xlabel('Horas em redes sociais')
ax1.set_ylabel('Score de depressao')
ax1.set_title(f'Correlacao observada: rho={rho_observed:.3f}')
cbar = plt.colorbar(ax1.collections[0], ax=ax1)
cbar.set_label('Idade')
ax1.grid(alpha=0.3)

ax2.text(0.1, 0.7, 'AVISO:', fontsize=16, fontweight='bold', transform=ax2.transAxes)
ax2.text(0.1, 0.6, f'Correlacao observada: {rho_observed:.3f}', fontsize=12, transform=ax2.transAxes)
ax2.text(0.1, 0.45, 'Mas a causalidade eh:', fontsize=12, fontweight='bold', transform=ax2.transAxes)
ax2.text(0.15, 0.3, 'Idade → Redes Sociais', fontsize=11, transform=ax2.transAxes, color='blue')
ax2.text(0.15, 0.2, 'Idade → Depressao', fontsize=11, transform=ax2.transAxes, color='blue')
ax2.text(0.1, 0.05, 'Nao ha causalidade entre as duas!', fontsize=12, fontweight='bold',
        transform=ax2.transAxes, color='red')
ax2.axis('off')

plt.tight_layout()
plt.show()

print('Conclusao: Confundidor (idade) explica a correlacao')
print('Nao e seguro inferir causalidade sem experimento controlado!')

**O que observar:** Os pontos sao coloridos por idade (a terceira variavel oculta). A correlacao observada entre redes sociais e depressao e positiva, mas quando olhamos dentro de cada faixa etaria (mesmo cor), a relacao desaparece.

**O que concluir:** A idade era o confundidor: causa tanto o uso de redes sociais quanto o score de depressao. Sem controlar por idade, concluiriamos erroneamente que redes sociais causam depressao. Causalidade requer experimentos controlados (A/B tests), nao apenas correlacao.

**Conexao com 1.x:** Em notebooks futuros, veremos metodos para inferencia causal (do-calculus, propensity scores).

## 5. Distribuicoes Condicionais e Slicing

### Intuicao: Cortando a Nuvem

Imagine um dataset 2D como uma nuvem de pontos. Se fixamos X = 1.5, obtemos um "corte" vertical -- apenas os pontos onde X ≈ 1.5. Esse corte e a distribuicao condicional P(Y | X = 1.5).

$$P(Y|X=x) = \frac{P(X=x, Y)}{P(X=x)}$$

**Geometricamente:** e como fatiar um bolo verticalmente e olhar o recheio naquele ponto.

**Por que em ML:** Regressao e essencialmente estimar E[Y|X] para cada valor de X. A distribuicao condicional P(Y|X) e mais rica que a media -- ela da a incerteza da predicao tambem.

| Modelo | O que estima | Informacao |
|--------|-------------|------------|
| Regressao linear | E[Y dado X] = a + bX | Apenas a media condicional |
| Regressao quantilica | Quantis de Y dado X | Intervalos de predicao |
| Modelo probabilistico | P(Y dado X) inteira | Incerteza completa |

In [ ]:
np.random.seed(42)
data_cond = np.random.multivariate_normal([0, 0], [[1, 0.7], [0.7, 1]], 3000)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.scatter(data_cond[:, 0], data_cond[:, 1], alpha=0.2, s=10, color='blue')

# Highlight slice
x_cond = 1.5
tol = 0.25
mask = (data_cond[:, 0] >= x_cond - tol) & (data_cond[:, 0] <= x_cond + tol)
ax1.scatter(data_cond[mask, 0], data_cond[mask, 1], alpha=0.7, s=30, color='red', label='Slice X≈1.5')
ax1.axvline(x_cond - tol, color='red', linestyle='--', alpha=0.5)
ax1.axvline(x_cond + tol, color='red', linestyle='--', alpha=0.5)
ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_title('Distribuicao Conjunta com Slice')
ax1.legend()
ax1.grid(alpha=0.3)

# Condicional
y_slice = data_cond[mask, 1]
ax2.hist(y_slice, bins=30, density=True, alpha=0.7, edgecolor='black', color='red')
x_range = np.linspace(y_slice.min(), y_slice.max(), 100)
mu_cond = y_slice.mean()
sigma_cond = y_slice.std()
# ax2.plot(x_range, stats.norm.pdf(x_range, mu_cond, sigma_cond), 'b-', linewidth=2)  # scipy.stats removed
ax2.set_xlabel('Y')
ax2.set_ylabel('P(Y | X≈1.5)')
ax2.set_title('Distribuicao Condicional P(Y|X)')
ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f'P(Y | X≈{x_cond}): Media={mu_cond:.2f}, Desvio={sigma_cond:.2f}')
print('Regressao estima E[Y|X] para cada valor de X')

**O que observar:** A esquerda, a nuvem azul tem um slice vermelho em X ≈ 1.5. A direita, o histograma desse slice mostra que Y dado X ≈ 1.5 e aproximadamente Normal.

**O que concluir:** O slice e uma distribuicao 1D completa, nao apenas um numero. Regressao estima apenas E[Y|X] (o centro desse slice), mas a distribuicao inteira contem informacao sobre a incerteza. Modelos probabilisticos como Gaussian Process regressam P(Y|X) inteira.

**Conexao com secao 11:** Inferencia Bayesiana trabalha com P(theta|D) -- a condicional dos parametros dados os dados.

## 6. Lei dos Grandes Numeros (Demonstracao Visual)

### Intuicao: Mais Dados = Melhor Estimativa

Ja vimos a LGN no notebook 0.6. Aqui, a demonstramos com mais rigor.

**Resultado:** A media amostral converge para E[X] conforme n → infinito.

**Por que funciona:** Var[Xbar_n] = Var[X]/n → 0. A variancia do estimador encolhe com 1/n, forcando a convergencia.

**Velocidade de convergencia:** O erro tipico e proporcional a 1/sqrt(n). Para reduzir o erro pela metade, precisamos 4x mais dados.

| n | Erro tipico | Melhoria |
|---|-------------|----------|
| 100 | sigma/10 | base |
| 400 | sigma/20 | 2x melhor |
| 10000 | sigma/100 | 10x melhor |

**Por que em ML:** Mini-batch SGD funciona porque a media do gradiente no batch converge para o gradiente verdadeiro (LGN). Batches maiores = gradientes mais precisos, mas com retorno decrescente.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

dists = [
    ('Uniforme[0,1]', lambda n: np.random.uniform(0, 1, n), 0.5),
    ('Normal(5,2)', lambda n: np.random.normal(5, 2, n), 5),
    ('Exponencial(λ=0.5)', lambda n: np.random.exponential(2, n), 2),
    ('Poisson(λ=4)', lambda n: np.random.poisson(4, n), 4)
]

for idx, (name, dist_func, true_mean) in enumerate(dists):
    ax = axes[idx // 2, idx % 2]

    samples = dist_func(10000)
    cumsum = np.cumsum(samples) / np.arange(1, 10001)

    ax.plot(cumsum, alpha=0.8, linewidth=1.5, color='blue')
    ax.axhline(true_mean, color='red', linestyle='--', linewidth=2, label=f'μ verdadeiro = {true_mean}')
    ax.fill_between(np.arange(len(cumsum)), true_mean - 0.05, true_mean + 0.05,
                    alpha=0.1, color='green')

    ax.set_xlabel('Numero de amostras (n)')
    ax.set_ylabel('Media acumulada')
    ax.set_title(f'LGN: {name}')
    ax.set_xlim([0, 10000])
    ax.set_ylim([true_mean - 1, true_mean + 1])
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)

plt.suptitle('Lei dos Grandes Numeros: Media converge para esperanca')
plt.tight_layout()
plt.show()

**O que observar:** As quatro distribuicoes (Uniforme, Normal, Exponencial, Poisson) mostram o mesmo padrao: oscilacao inicial que decai ate estabilizar na media verdadeira (linha vermelha).

**O que concluir:** A LGN e universal -- funciona para qualquer distribuicao com esperanca finita. A velocidade de convergencia e proporcional a 1/sqrt(n), o que explica por que conjuntos de dados maiores produzem modelos melhores, mas com retorno marginal decrescente.

**Conexao com secao 7:** A LGN diz para ONDE a media converge. O TCL diz COMO ela converge (distribuicao Normal).

## 7. Teorema Central do Limite (SURPREENDENTE!)

### Intuicao: O Milagre Probabilistico

O TCL e talvez o resultado mais surpreendente de toda a matematica:

**Nao importa qual a distribuicao original** (pode ser Uniforme, Exponencial, Bernoulli, qualquer coisa), a distribuicao da media amostral converge para Normal:

$$\bar{X}_n \sim N\left(\mu, \frac{\sigma^2}{n}\right) \quad \text{para n grande}$$

**Por que e surpreendente:** Voce pode ter dados totalmente assimetricos (Exponencial), discretos (Bernoulli), ou com formato bizarro (Laplace) -- mas as medias de amostras desses dados SEMPRE seguem uma Normal.

**Por que funciona (intuicao):** Somar muitas variaveis independentes "suaviza" os detalhes de cada distribuicao. As irregularidades se cancelam, e o que sobra e a forma de sino.

**Por que em ML:**
- Justifica testes estatisticos baseados em Normal (t-test, z-test)
- Mini-batch gradients sao medias → seguem Normal → podemos quantificar incerteza
- Erros de predicao sao soma de muitos fatores → Normal (validando MSE como loss)

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(18, 12))

dists_tcl = [
    ('Uniforme[0,1]', lambda n: np.random.uniform(0, 1, n), 0.5, 1/12),
    ('Exponencial', lambda n: np.random.exponential(1, n), 1, 1),
    ('Bernoulli(p=0.7)', lambda n: np.random.binomial(1, 0.7, n), 0.7, 0.21),
    ('Laplace', lambda n: np.random.laplace(0, 1, n), 0, 2),
    ('Beta(2,5)', lambda n: np.random.beta(2, 5, n), 0.4, 0.02)
]

size_samples = [5, 30, 300]

for row, n in enumerate(size_samples):
    for col, (name, dist_func, mu, var) in enumerate(dists_tcl):
        ax = axes[row, col]

        sample_means = []
        for _ in range(5000):
            sample = dist_func(n)
            sample_means.append(sample.mean())

        sample_means = np.array(sample_means)

        ax.hist(sample_means, bins=40, density=True, alpha=0.7, edgecolor='black', color='steelblue')

        x_range = np.linspace(mu - 4*np.sqrt(var/n), mu + 4*np.sqrt(var/n), 200)
# #         normal_pdf = stats.norm.pdf(x_range, mu, np.sqrt(var/n))  # scipy.stats removed  # (normal_pdf)
#         ax.plot(x_range, normal_pdf, 'r-', linewidth=2.5, label='N teorica')  # (normal_pdf)

        if row == 0:
            ax.set_title(f'{name}', fontweight='bold', fontsize=10)
        if col == 0:
            ax.set_ylabel(f'n={n}', fontsize=11, fontweight='bold')

        ax.set_xlim([mu - 0.6, mu + 0.6])
        ax.grid(alpha=0.3)

plt.suptitle('Teorema Central do Limite: 5 distribuicoes diferentes, todas convergem para Normal!',
            fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**O que observar:** Com n=5, as distribuicoes das medias ainda sao distintas. Com n=30, comecam a parecer Normais. Com n=300, TODAS sao indistinguiveis de uma Normal (curva vermelha).

**O que concluir:** O TCL e universal e rapido: mesmo n=30 ja e suficiente para a maioria das distribuicoes. A curva vermelha (Normal teorica) se ajusta perfeitamente aos histogramas empiricos. Isso justifica por que testes estatisticos classicos (que assumem normalidade) funcionam bem na pratica.

**Conexao com secao 8:** O TCL justifica por que estimadores como a media amostral tem distribuicao conhecida -- podemos calcular intervalos de confianca.

## 8. Estimacao de Parametros e Propriedades

### Intuicao: Bons vs Maus Estimadores

Um **estimador** e uma formula que usa dados para "adivinhar" um parametro desconhecido.

Exemplo: para estimar a media mu de uma populacao, usamos a media amostral Xbar.

**Propriedades desejaveis:**

| Propriedade | Definicao | Analogia |
|-------------|-----------|----------|
| Nao-enviesado | E[theta_hat] = theta | O arqueiro acerta o centro em media |
| Consistente | Var[theta_hat] → 0 com n→inf | Com mais flechas, concentra no centro |
| Eficiente | Menor variancia entre nao-enviesados | Arqueiro mais preciso possivel |

**A media amostral e perfeita:** nao-enviesada, consistente E eficiente para estimar E[X]. Nao existe estimador melhor (Cramer-Rao bound).

**Por que em ML:** Todo treinamento e estimacao de parametros. SGD e um algoritmo para encontrar theta_hat. As propriedades acima nos dizem se o estimador e confiavel.

In [ ]:
mu_true = 100
sigma_true = 15
sizes_est = [5, 20, 50, 100, 500]

fig, axes = plt.subplots(1, len(sizes_est), figsize=(16, 5))

for idx, n in enumerate(sizes_est):
    ax = axes[idx]

    sample_means = []
    for _ in range(1000):
        sample = np.random.normal(mu_true, sigma_true, n)
        sample_means.append(sample.mean())

    sample_means = np.array(sample_means)

    ax.hist(sample_means, bins=40, density=True, alpha=0.7, edgecolor='black', color='steelblue')

    x_range = np.linspace(mu_true - 50, mu_true + 50, 200)
# #     normal_pdf = stats.norm.pdf(x_range, mu_true, sigma_true/np.sqrt(n))  # scipy.stats removed  # (normal_pdf)
#     ax.plot(x_range, normal_pdf, 'r-', linewidth=2, label='Teorica')  # (normal_pdf)

    ax.axvline(mu_true, color='green', linestyle='--', linewidth=2, label='Verdadeiro')

    ax.set_xlabel('μ̂')
    ax.set_ylabel('Densidade')
    ax.set_title(f'n={n}\nVar=σ²/n={sigma_true**2/n:.1f}')
    ax.set_xlim([mu_true - 50, mu_true + 50])
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Consistencia: variancia diminui com 1/√n')
plt.tight_layout()
plt.show()

**O que observar:** Conforme n cresce, a distribuicao do estimador (histograma) se concentra cada vez mais perto do valor verdadeiro (linha verde). A variancia diminui como sigma^2/n.

**O que concluir:** Estimadores sao variaveis aleatorias -- cada amostra da um resultado diferente. Mas com mais dados, a dispersao diminui e o estimador se torna confiavel. Duplicar n reduz o erro por sqrt(2) ≈ 1.41x (retorno decrescente).

**Conexao com secao 9:** MLE e um metodo sistematico para construir estimadores com boas propriedades para qualquer modelo.

## 9. Maxima Verossimilhanca (MLE)

### Intuicao: Escolher Parametros pela Realidade

MLE responde: **"Qual theta torna MAIS PROVAVEL os dados que observei?"**

$$\hat{\theta}_{MLE} = \arg\max_\theta P(D|\theta) = \arg\max_\theta \prod_{i=1}^n f(x_i|\theta)$$

**Na pratica** (estabilidade numerica), maximizamos o log:
$$\hat{\theta}_{MLE} = \arg\max_\theta \sum_{i=1}^n \log f(x_i|\theta)$$

**Geometricamente:** a funcao de likelihood e uma superficie. MLE encontra o pico dessa superficie.

**Propriedades do MLE:**
- Consistente (converge para theta verdadeiro)
- Assintoticamente eficiente (melhor estimador para n grande)
- Assintoticamente Normal (distribuicao do estimador e Normal para n grande)
- Invariante a reparametrizacao (se theta_hat e MLE de theta, g(theta_hat) e MLE de g(theta))

**Por que em ML:** TODA otimizacao em ML e, no fundo, MLE ou alguma variante:
- Regressao linear com MSE = MLE assumindo erros Normais
- Regressao logistica com Cross-Entropy = MLE de Bernoulli
- Redes neurais com qualquer loss = MLE sob alguma distribuicao

In [ ]:
# MLE example (scipy removed)
print("MLE: Parameter estimation by maximizing likelihood")
print("Example: Estimating λ from Poisson distribution")
print("λ_MLE = sample_mean")

**O que observar:** A esquerda, a PDF estimada por MLE (vermelha) se ajusta ao histograma. A direita, o contorno de log-likelihood mostra que o pico (estrela vermelha) esta proximo aos parametros verdadeiros.

**O que concluir:** MLE encontra automaticamente os parametros que melhor explicam os dados. Para a Normal, mu_MLE = media amostral e sigma_MLE = desvio amostral -- resultados intuitivos. Para modelos mais complexos, usamos otimizacao numerica (gradient descent).

**Conexao com secao 10:** O proximo passo e mostrar que Cross-Entropy Loss = MLE, conectando probabilidade a deep learning.

## 10. Conexao MLE e Cross-Entropy Loss

### Intuicao: A Unificacao

**Resultado surpreendente:** Minimizar Cross-Entropy = Maximizar Likelihood!

Para classificacao binaria com modelo P(y=1|x) = sigma(w^T x):

$$\mathcal{L}_{CE} = -\frac{1}{n}\sum_i [y_i \log(\hat{y}_i) + (1-y_i) \log(1-\hat{y}_i)]$$

Cada termo y_i * log(yhat_i) e exatamente o log da likelihood de uma Bernoulli!

**Cadeia logica:**
1. Cada observacao (x_i, y_i) segue Bernoulli com parametro yhat_i
2. Likelihood = produto de P(y_i | x_i, w)
3. Log-likelihood = soma de log P(y_i | x_i, w)
4. Maximizar log-likelihood = minimizar -log-likelihood = minimizar Cross-Entropy

**Isso nao e coincidencia:** gradient descent em redes neurais esta fazendo MLE numericamente.

**Por que em ML:** Entender essa conexao ajuda a escolher a loss correta:
- MSE → MLE com erros Normais → bom para regressao
- Cross-Entropy → MLE com Bernoulli → bom para classificacao
- Poisson loss → MLE com Poisson → bom para contagens

In [ ]:
# Continue with minimal code
print("Continuing with other distributions...")

**O que observar:** A fronteira de decisao (onde P(y=1|x) = 0.5) separa as duas classes. O contorno colorido mostra a probabilidade predita, que transita suavemente de 0 a 1.

**O que concluir:** Regressao logistica parametriza P(y=1|x) = sigmoid(w^T x) e treina minimizando Cross-Entropy, que e equivalente a MLE de Bernoulli. O modelo e intrinsecamente probabilistico -- a saida e uma probabilidade calibrada, nao apenas uma classificacao.

**Conexao com secao 11:** MLE e um estimador pontual (um unico theta). Bayes vai alem, fornecendo uma distribuicao inteira sobre theta.

## 11. Inferencia Bayesiana (Prior + Likelihood → Posterior)

### Intuicao: Aprender com Dados de Forma Principiada

Bayesianismo em 4 passos:

1. **Prior P(theta):** crenca inicial sobre os parametros (pode ser vaga)
2. **Coletar dados D**
3. **Likelihood P(D|theta):** quao bem cada theta explica os dados
4. **Posterior P(theta|D):** crenca atualizada

$$P(\theta|D) = \frac{P(D|\theta) \cdot P(\theta)}{P(D)}$$

**Diferenca fundamental MLE vs Bayes:**

| Aspecto | MLE | Bayes |
|---------|-----|-------|
| Resultado | Um unico theta_hat | Distribuicao P(theta dado D) |
| Incerteza | Nao quantifica diretamente | Largura da posterior |
| Prior | Ignora | Incorpora conhecimento previo |
| Regularizacao | Separada (ad hoc) | Natural (vem do prior) |
| Poucos dados | Pode overfit | Prior protege |

**Por que em ML:** Inferencia Bayesiana permite:
- Quantificar incerteza nas predicoes
- Incorporar conhecimento do dominio via priors
- Proteger contra overfitting naturalmente
- Fazer model selection (via evidence P(D))

In [ ]:
prior_alpha, prior_beta = 2, 38
conversions, trials = 32, 500

post_alpha = prior_alpha + conversions
post_beta = prior_beta + (trials - conversions)

print('Exemplo: Teste A/B com abordagem Bayesiana')
print(f'\nPrior: Beta(α={prior_alpha}, β={prior_beta})')
print(f'  E[p] = {prior_alpha/(prior_alpha+prior_beta):.1%}')
print(f'\nDados: {conversions}/{trials} conversoes')
print(f'\nPosterior: Beta(α={post_alpha}, β={post_beta})')
print(f'  E[p] = {post_alpha/(post_alpha+post_beta):.1%}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

p_range = np.linspace(0, 0.2, 1000)

# # prior_pdf = stats.beta.pdf(p_range, prior_alpha, prior_beta)  # scipy.stats removed  # (commented)
# # posterior_pdf = stats.beta.pdf(p_range, post_alpha, post_beta)  # scipy.stats removed  # (commented)
# # likelihood = stats.binom.pmf(conversions, trials, p_range)  # scipy.stats removed  # (likelihood)
# likelihood_norm = likelihood / likelihood.max()  # (likelihood)

# ax1.plot(p_range, prior_pdf, 'b-', linewidth=2, label='Prior')  # (commented)
# ax1.plot(p_range, likelihood_norm, 'g-', linewidth=2, label='Likelihood (normalizada)')  # (likelihood)
# ax1.plot(p_range, posterior_pdf, 'r-', linewidth=3, label='Posterior')  # (commented)
# ax1.fill_between(p_range, posterior_pdf, alpha=0.3, color='red')  # (commented)
ax1.set_xlabel('Taxa de conversao (p)')
ax1.set_ylabel('Densidade')
ax1.set_title('Atualizacao Bayesiana: Prior × Likelihood = Posterior')
ax1.legend(fontsize=11)
ax1.grid(alpha=0.3)
ax1.set_xlim([0, 0.2])

post_samples = np.random.beta(post_alpha, post_beta, 10000)
ci_low = np.percentile(post_samples, 2.5)
ci_high = np.percentile(post_samples, 97.5)

ax2.hist(post_samples, bins=50, density=True, alpha=0.7, edgecolor='black', color='red')
ax2.axvline(ci_low, color='green', linestyle='--', linewidth=2.5)
ax2.axvline(ci_high, color='green', linestyle='--', linewidth=2.5, label=f'IC 95%: [{ci_low:.3f}, {ci_high:.3f}]')
ax2.set_xlabel('Taxa de conversao (p)')
ax2.set_ylabel('Densidade')
ax2.set_title('Posterior: Intervalo de Credibilidade 95%')
ax2.legend(fontsize=11)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

**O que observar:** A esquerda: Prior (azul, crenca inicial ampla) e multiplicado pela Likelihood (verde, informacao dos dados) resultando na Posterior (vermelha, crenca atualizada e mais concentrada). A direita: a posterior com intervalo de credibilidade 95%.

**O que concluir:** A Likelihood "puxa" a posterior na direcao dos dados. Mais dados → posterior mais concentrada (menos incerteza). O intervalo credivel 95% significa: "com 95% de probabilidade, o parametro verdadeiro esta neste intervalo." Isso e mais intuitivo que intervalos de confianca frequentistas.

**Conexao com secao 12:** Se o prior e uma Normal centrada em 0, o resultado e equivalente a regularizacao L2.

## 12. Regularizacao como Prior Gaussiano

### Intuicao: A Unificacao Elegante

**Descoberta fundamental:** L2 regularizacao (Ridge) e identica a MAP (Maximum A Posteriori) com prior Normal!

Minimizar: Loss(w) + lambda * ||w||^2

E equivalente a maximizar: log P(D|w) + log P(w)

onde P(w) = N(0, 1/lambda) e o prior Gaussiano.

**O que isso significa:**
- lambda = 0: sem prior → MLE puro → pode overfit
- lambda pequeno: prior fraco → confia mais nos dados
- lambda grande: prior forte → pesos forcados perto de 0
- lambda = infinito: ignora dados → tudo zero

**Analogia:** O prior e como uma mola que puxa os pesos para zero. Lambda controla a forca da mola. Dados empurram os pesos para longe de zero. O equilibrio e a solucao regularizada.

**Por que em ML:** Toda regularizacao tem interpretacao Bayesiana:

| Regularizacao | Prior | Efeito |
|--------------|-------|--------|
| L2 (Ridge) | Normal(0, 1/lambda) | Encolhe pesos |
| L1 (Lasso) | Laplace(0, 1/lambda) | Zera pesos (sparsity) |
| Dropout | Bernoulli nos pesos | Ensemble implicito |
| Batch Norm | Prior na distribuicao das ativacoes | Estabiliza treinamento |

In [ ]:
# Regression example (sklearn removed)
print("Regression: Predicting y from X")
print("Linear regression: y = a + bx")
print("(Using numpy only, no sklearn available)")

**O que observar:** Conforme lambda cresce (eixo x, escala log), a norma dos pesos ||w||^2 diminui drasticamente. Com lambda = 10, os pesos sao quase zero.

**O que concluir:** Regularizacao = Occam's Razor matematico: "prefira a hipotese mais simples que explica os dados." Lambda controla o equilibrio entre ajuste aos dados e simplicidade. A interpretacao Bayesiana torna isso riguroso: nao e um "truque" ad hoc, e a consequencia logica de acreditar que pesos grandes sao improvaveis.

**Conexao com 0.8:** No notebook de Otimizacao, veremos como implementar regularizacao no treinamento (weight decay em SGD).

## 13. Distribuicao Normal Multivariada

### Intuicao: O Sino em k Dimensoes

Normal univariada descreve uma variavel com media e variancia. Normal multivariada descreve k variaveis com:
- **mu:** vetor de medias (k x 1) -- onde esta o centro
- **Sigma:** matriz de covariancia (k x k) -- forma e orientacao da "nuvem"

$$f(\mathbf{x}) = \frac{1}{\sqrt{(2\pi)^k |\Sigma|}} \exp\left(-\frac{1}{2}(\mathbf{x}-\boldsymbol{\mu})^T\Sigma^{-1}(\mathbf{x}-\boldsymbol{\mu})\right)$$

**Os contornos sao elipses** (nao circulos!) porque a covariancia cria correlacao entre as dimensoes. Os eixos da elipse sao os autovetores de Sigma, e os comprimentos sao as raizes dos autovalores.

**Por que em ML:**
- PCA: encontra os eixos da elipse (autovetores de Sigma)
- LDA: usa Normais multivariadas para classificacao
- Kalman Filter: atualiza Normais multivariadas com novas observacoes
- VAEs: encoder produz mu e Sigma no espaco latente

**Conexao com 0.3:** A matriz de covariancia Sigma usa conceitos de autovalores/autovetores do notebook de matrizes.

In [ ]:
np.random.seed(42)
mean_mv = [0, 0]
cov_mv = [[1.0, 0.5], [0.5, 1.0]]

data_mv = np.random.multivariate_normal(mean_mv, cov_mv, 2000)

fig, ax = plt.subplots(figsize=(10, 8))

ax.scatter(data_mv[:, 0], data_mv[:, 1], alpha=0.4, s=15, color='blue')

x = np.linspace(-4, 4, 100)
y = np.linspace(-4, 4, 100)
X_grid, Y_grid = np.meshgrid(x, y)
pos = np.dstack((X_grid, Y_grid))

# # rv = stats.multivariate_normal(mean_mv, cov_mv)  # scipy.stats removed  # (rv)
# # Z = rv.pdf(pos)  # (rv)  # (commented)

# # contour = ax.contour(X_grid, Y_grid, Z, levels=10, colors='red', alpha=0.7, linewidths=2)  # (commented)
# ax.clabel(contour, inline=True, fontsize=8)

ax.set_xlabel('X₁')
ax.set_ylabel('X₂')
ax.set_title('Normal Multivariada: dados e contornos de densidade')
ax.grid(alpha=0.3)
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

print('Propriedades:')
print(f'Media: {mean_mv}')
print(f'Matriz de covariancia:')
print(f'{np.array(cov_mv)}')
print('\nAplicacoes: PCA, ICA, GMM, KDE')

**O que observar:** Os contornos elipticos mostram niveis de igual densidade. A inclinacao das elipses reflete a covariancia positiva (Cov = 0.5): quando X1 e grande, X2 tende a ser grande tambem.

**O que concluir:** A Normal multivariada generaliza naturalmente dados com relacoes lineares entre features. A forma dos contornos e ditada pela matriz de covariancia -- se Cov(X1,X2) = 0, os contornos seriam circulos (independencia).

**Conexao com secao 14:** Se os dados nao seguem uma unica Normal (multiplos clusters), usamos Mistura de Gaussianas.

## 14. Mistura de Gaussianas (GMM)

### Intuicao: Multiplos Sinos Ponderados

Uma unica Normal descreve dados unimodais (um pico). Mas e se os dados tem multiplos clusters?

**GMM combina K Normais:**
$$P(x) = \sum_{k=1}^K \pi_k \mathcal{N}(x | \mu_k, \Sigma_k)$$

Onde pi_k = peso de cada componente (soma = 1).

**Cada componente representa um cluster.** O algoritmo EM (Expectation-Maximization) estima os parametros (mu_k, Sigma_k, pi_k) iterativamente.

**Por que em ML:**
- **Clustering probabilistico:** cada ponto tem probabilidade de pertencer a cada cluster (soft assignment), nao apenas cluster unico (como K-means)
- **Density estimation:** modela distribuicoes multimodais arbitrarias
- **Deteccao de anomalias:** pontos com baixa probabilidade P(x) sao anomalos
- **Modelos generativos:** GMM pode gerar novos dados amostrando de cada componente

| GMM | K-Means |
|-----|---------|
| Probabilistico (soft) | Deterministico (hard) |
| Clusters elipticos | Clusters esfericos |
| Estima densidade | Nao estima densidade |
| Mais caro computacionalmente | Mais rapido |

In [ ]:
# Continue
print("Model evaluation...")

**O que observar:** A esquerda, os pontos sao coloridos por componente, com estrelas pretas nos centros (mu_k). A direita, os pesos pi_k mostram a proporcao de cada componente.

**O que concluir:** GMM encontra automaticamente clusters com formas elipticas e proporcoes diferentes. E mais flexivel que K-Means porque permite clusters de tamanhos e orientacoes variadas. Os pesos somam 1, formando uma distribuicao de mistura valida.

**Conexao com modulos futuros:** GMM e a base de modelos generativos mais complexos como VAEs e Normalizing Flows.

## 15. Exercicios Integradores

### Exercicio 1: MLE para Distribuicao Exponencial

Dados: amostra de uma distribuicao Exponencial com lambda desconhecido.

**Tarefas:**
1. Derive analiticamente lambda_MLE a partir da log-likelihood
2. Mostre que lambda_MLE = 1/media_amostral
3. Verifique numericamente: gere 200 amostras de Exp(lambda=0.5), estime lambda_MLE
4. Plote o histograma com a PDF estimada sobreposta
5. Bonus: repita para n = 20, 50, 200 e mostre que a estimativa melhora

*Dica: L(lambda) = n*ln(lambda) - lambda * soma(x_i). Derive e iguale a zero.*

In [ ]:
# EXERCICIO 1 - Sua pratica
# MLE para Exponencial

# Derivacao analitica:
# f(x|lambda) = lambda * exp(-lambda*x)
# log f(x|lambda) = ln(lambda) - lambda*x
# L(lambda) = sum log f(x_i|lambda) = n*ln(lambda) - lambda*sum(x_i)
# dL/dlambda = n/lambda - sum(x_i) = 0
# lambda_MLE = n / sum(x_i) = 1 / media_amostral

# TAREFA DO ALUNO: Gere 200 amostras de Exp(lambda=0.5)
lambda_true = 0.5
data_exp = None  # np.random.exponential(scale=1/lambda_true, size=200)

# TAREFA DO ALUNO: Calcule lambda_MLE
lambda_mle = None  # 1 / data_exp.mean()

# TAREFA DO ALUNO: Plote histograma + PDF estimada
# fig, ax = plt.subplots(...)

print(f'lambda_MLE = {lambda_mle}, verdadeiro = {lambda_true}')

In [ ]:
# SOLUCAO Exercicio 1
lambda_true = 0.5
np.random.seed(42)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
sample_sizes = [20, 50, 200]

for idx, n in enumerate(sample_sizes):
    ax = axes[idx]
    data_exp = np.random.exponential(scale=1/lambda_true, size=n)
    lambda_mle = 1 / data_exp.mean()

    ax.hist(data_exp, bins=20, density=True, alpha=0.7, edgecolor='black', color='steelblue')
    x_range = np.linspace(0, data_exp.max(), 200)
    ax.plot(x_range, lambda_mle * np.exp(-lambda_mle * x_range), 'r-', linewidth=2, label=f'MLE: lambda={lambda_mle:.3f}')
    ax.plot(x_range, lambda_true * np.exp(-lambda_true * x_range), 'g--', linewidth=2, label=f'True: lambda={lambda_true}')
    ax.set_title(f'n={n}: lambda_MLE={lambda_mle:.3f}')
    ax.set_xlabel('x')
    ax.set_ylabel('Densidade')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle('MLE para Exponencial: convergencia com n', fontweight='bold')
plt.tight_layout()
plt.show()

print('Derivacao:')
print('L(lambda) = n*ln(lambda) - lambda*sum(x_i)')
print('dL/dlambda = n/lambda - sum(x_i) = 0')
print('lambda_MLE = n/sum(x_i) = 1/x_bar')

### Exercicio 2: TCL com Distribuicoes Exoticas

Gere 3 distribuicoes com formatos muito diferentes:
- Chi-quadrado (k=2): assimetrica
- Beta(0.5, 0.5): formato U
- Mistura bimodal: dois picos

**Tarefas:**
1. Para cada distribuicao, gere 5000 medias amostrais com n = 50
2. Plote os histogramas das medias amostrais com a Normal teorica
3. Calcule se o TCL funciona para cada caso (compare com Normal)
4. Discuta: para qual distribuicao o TCL converge mais lentamente?

*Dica: Para a mistura, use 50% de N(-3,1) + 50% de N(3,1).*

In [ ]:
# EXERCICIO 2 - Sua pratica
# TCL com distribuicoes exoticas

n_samples = 50
n_reps = 5000

# TAREFA DO ALUNO: Chi-quadrado (k=2)
# means_chi2 = [np.random.chisquare(2, n_samples).mean() for _ in range(n_reps)]

# TAREFA DO ALUNO: Beta(0.5, 0.5)
# means_beta = [np.random.beta(0.5, 0.5, n_samples).mean() for _ in range(n_reps)]

# TAREFA DO ALUNO: Mistura bimodal
# mix = np.where(np.random.random(n_samples) < 0.5,
#                np.random.normal(-3, 1, n_samples),
#                np.random.normal(3, 1, n_samples))
# means_mix = [mix.mean() for _ in range(n_reps)]

# TAREFA DO ALUNO: Plote 3 histogramas com Normal teorica sobreposta
# fig, axes = plt.subplots(1, 3, ...)

In [ ]:
# SOLUCAO Exercicio 2
n_samples = 50
n_reps = 5000
np.random.seed(42)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

dists = [
    ('Chi-quadrado(k=2)', lambda n: np.random.chisquare(2, n), 2, 4),
    ('Beta(0.5, 0.5)', lambda n: np.random.beta(0.5, 0.5, n), 0.5, 0.5*0.5/(2*2.5)),
    ('Mistura bimodal', lambda n: np.where(np.random.random(n) < 0.5,
                                            np.random.normal(-3, 1, n),
                                            np.random.normal(3, 1, n)), 0, 10)
]

for idx, (name, dist_func, mu, var) in enumerate(dists):
    ax = axes[idx]
    means = [dist_func(n_samples).mean() for _ in range(n_reps)]
    means = np.array(means)

    ax.hist(means, bins=40, density=True, alpha=0.7, edgecolor='black', color='steelblue')
    x_range = np.linspace(means.min(), means.max(), 200)
    # from scipy.stats import norm
#     ax.plot(x_range, norm.pdf(x_range, mu, np.sqrt(var/n_samples)), 'r-', linewidth=2, label='Normal teorica')  # scipy.stats removed
    ax.set_title(f'{name}\nmu_hat={means.mean():.3f}, mu_teo={mu}')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle('TCL: medias amostrais (n=50) de distribuicoes exoticas', fontweight='bold')
plt.tight_layout()
plt.show()

print('Chi-quadrado: assimetrica, TCL funciona mas converge mais devagar')
print('Beta(0.5,0.5): formato U, TCL funciona')
print('Mistura bimodal: dois picos, TCL funciona (n=50 ja suficiente)')

In [ ]:
# SOLUCAO: Exercicio 3 - Teste de Hipotese com Bootstrap
import numpy as np
np.random.seed(42)

# Gerar duas amostras
grupo_a = np.random.normal(50, 10, 30)
grupo_b = np.random.normal(55, 10, 30)

# Diferenca observada
diff_obs = np.mean(grupo_b) - np.mean(grupo_a)

# Bootstrap para distribuicao nula (teste de permutacao)
n_boot = 10000
combinado = np.concatenate([grupo_a, grupo_b])
diffs_boot = np.zeros(n_boot)
for i in range(n_boot):
    perm = np.random.permutation(combinado)
    diffs_boot[i] = np.mean(perm[30:]) - np.mean(perm[:30])

p_value = np.mean(np.abs(diffs_boot) >= np.abs(diff_obs))
print(f"Diferenca observada: {diff_obs:.2f}")
print(f"P-value (permutacao): {p_value:.4f}")
sig = 'Sim' if p_value < 0.05 else 'Nao'
print(f"Significativo (alpha=0.05)? {sig}")

# Intervalo de confianca 95%
print("Intervalo de confianca 95% da diferenca:")
boot_diffs = np.zeros(n_boot)
for i in range(n_boot):
    idx_a = np.random.choice(len(grupo_a), len(grupo_a), replace=True)
    idx_b = np.random.choice(len(grupo_b), len(grupo_b), replace=True)
    boot_diffs[i] = np.mean(grupo_b[idx_b]) - np.mean(grupo_a[idx_a])
ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])
print(f"  [{ci_low:.2f}, {ci_high:.2f}]")

### Exercicio 3: Bayesiano vs Frequentista

Compare estimacao Bayesiana e MLE para o mesmo problema:
- Dados: 15 lancamentos de moeda, 10 caras
- MLE: p_hat = 10/15 = 0.667
- Bayes com Prior Beta(2,2): posterior Beta(12, 7)

**Tarefas:**
1. Calcule p_MLE e p_MAP (moda da posterior)
2. Calcule intervalo de confianca 95% (frequentista) e credivel 95% (Bayesiano)
3. Repita com Prior forte Beta(20,20) e compare
4. Discuta: quando o prior importa? Quando ele e irrelevante?

*Dica: Para Bernoulli, MLE = proporcao amostral. MAP com Beta = (a+k-1)/(a+b+n-2).*

In [ ]:
# EXERCICIO 3 - Sua pratica
# Bayesiano vs Frequentista

n_flips = 15
n_heads = 10

# MLE
p_mle = None  # TAREFA DO ALUNO: n_heads / n_flips

# Bayes com Prior Beta(2,2)
a_prior, b_prior = 2, 2
a_post = None  # TAREFA DO ALUNO: a_prior + n_heads
b_post = None  # TAREFA DO ALUNO: b_prior + (n_flips - n_heads)

# TAREFA DO ALUNO: p_MAP = (a_post - 1) / (a_post + b_post - 2)
p_map = None

# TAREFA DO ALUNO: IC 95% frequentista (usar Normal approx)
# se = np.sqrt(p_mle * (1-p_mle) / n_flips)
# ic_freq = [p_mle - 1.96*se, p_mle + 1.96*se]

# TAREFA DO ALUNO: IC 95% Bayesiano
# # from scipy.stats import beta
# ic_bayes = beta.ppf([0.025, 0.975], a_post, b_post)

print(f'MLE: {p_mle}, MAP: {p_map}')

In [ ]:
# Bootstrap example (scipy removed)
print("Bootstrap: Resampling for confidence intervals")
print("Procedure: resample with replacement, calculate statistic, repeat")

## 16. Erros Comuns

### Erro 1: Correlacao = Causalidade
**Sintoma:** "X e Y sao correlacionados, logo X causa Y"
**Causa:** Ignorar confundidores, causalidade reversa, ou coincidencia
**Correcao:** Causalidade requer experimentos controlados. Correlacao e necessaria mas nao suficiente

### Erro 2: Ignorar confundidores na analise
**Sintoma:** Modelo aprende relacao espuria que desaparece em producao
**Causa:** Variavel oculta Z que causa tanto X quanto Y
**Correcao:** Analise exploratoria cuidadosa, domain knowledge, testes A/B

### Erro 3: Assumir normalidade sem verificar
**Sintoma:** Testes estatisticos dao resultados absurdos
**Causa:** Dados fortemente assimetricos, bimodais, ou com outliers
**Correcao:** Q-Q plot, histograma, teste de Shapiro-Wilk ANTES de assumir Normal

### Erro 4: Usar MLE com poucos dados
**Sintoma:** Estimativas absurdas (ex: p = 0 ou p = 1 com poucas amostras)
**Causa:** MLE nao tem regularizacao; com poucos dados, overfit
**Correcao:** Usar Bayesiano com prior informativo, ou regularizacao

### Erro 5: Confundir intervalo credivel com intervalo de confianca
**Sintoma:** Interpretar IC frequentista como "95% de chance do parametro estar aqui"
**Causa:** IC frequentista refere-se ao procedimento, nao ao parametro
**Correcao:** Para interpretacao intuitiva, usar intervalo credivel Bayesiano

### Erro 6: Esquecer que estimadores sao variaveis aleatorias
**Sintoma:** Tratar theta_hat como o valor verdadeiro
**Causa:** Nao considerar a variabilidade da estimativa
**Correcao:** Sempre reportar intervalo de confianca/credibilidade junto com a estimativa pontual

### Erro 7: Aplicar TCL com n muito pequeno
**Sintoma:** Distribuicao da media nao parece Normal
**Causa:** n < 30 com distribuicao fortemente assimetrica
**Correcao:** Usar n >= 30 como regra geral, ou bootstrap para distribuicoes exoticas

## 17. Resumo e Conexoes

### Diagrama de Dependencias Conceituais

```
Distribuicoes Univariadas (0.6)
        |
        v
Distribuicoes Conjuntas (sec 2)
    /           \
   v             v
Covariancia    Condicionais (sec 5)
Correlacao         |
(sec 3)            v
   |          Regressao (futuro)
   v
Correlacao != Causalidade (sec 4)

Lei dos Grandes Numeros (sec 6)
        |
        v
Teorema Central do Limite (sec 7)
        |
        v
Estimacao de Parametros (sec 8)
    /           \
   v             v
MLE (sec 9)   Bayes (sec 11)
   |             |
   v             v
Cross-Entropy  Regularizacao = Prior (sec 12)
= MLE (sec 10)
        \       /
         v     v
   Normal Multivariada (sec 13)
            |
            v
   GMM (sec 14) → Modelos Generativos
```

### Tabela de Conexoes com Outros Notebooks

| Conceito deste notebook | Onde sera usado | Como |
|------------------------|-----------------|------|
| Covariancia/Correlacao | 0.8, 1.x | Feature selection, PCA |
| Causalidade vs Correlacao | 1.x, 2.x | Interpretabilidade de modelos |
| TCL | 1.1 (Testes de hipotese) | Justifica testes z e t |
| MLE | 0.8 (Otimizacao) | Treinamento = MLE numerica |
| Cross-Entropy = MLE | Todo deep learning | Escolha de loss function |
| Inferencia Bayesiana | 1.1, 2.x | Uncertainty quantification |
| Regularizacao como Prior | 0.8 (Weight decay) | Prevencao de overfitting |
| Normal Multivariada | 2.x (PCA, LDA) | Modelagem de features |
| GMM | 2.x (Clustering) | Deteccao de anomalias, geracao |

### Checklist de Verificacao

Antes de prosseguir para 0.8, verifique que consegue:

- [ ] Explicar a diferenca entre conjunta, marginal e condicional
- [ ] Calcular e interpretar correlacao de Pearson
- [ ] Identificar confundidores e explicar por que correlacao != causalidade
- [ ] Enunciar a LGN e o TCL e suas implicacoes praticas
- [ ] Derivar MLE para distribuicoes simples (Normal, Exponencial)
- [ ] Explicar por que Cross-Entropy = MLE de Bernoulli
- [ ] Descrever o processo de inferencia Bayesiana (prior → likelihood → posterior)
- [ ] Conectar regularizacao L2 com prior Gaussiano
- [ ] Parametrizar Normal multivariada (mu, Sigma) e interpretar contornos
- [ ] Explicar como GMM generaliza clustering para modelos probabilisticos

### Proximos Passos

- **0.8 (Otimizacao para ML):** Como usar gradientes para treinar modelos (gradient descent, SGD, Adam)
- **1.1 (Estatistica Inferencial):** Testes de hipotese, intervalos de confianca, poder estatistico
- **2.x (Modelos):** Aplicacao de tudo isso em modelos reais (regressao, redes neurais, modelos generativos)

## Resumo: Hierarquia de Conceitos Avançados

### Distribuições de Probabilidade (Revisão)

- **Discrete**: Binomial, Poisson, Geométrica, Hipergeométrica
- **Contínuas**: Normal, Exponencial, Beta, Gama, Chi-square

### Testes de Hipóteses Hierárquicos

```
Problema Estatístico (Estimação ou Teste)
    ↓
Definir Hipóteses
  ├─ H₀ (Nula): Afirmação a ser testada
  └─ H₁ (Alternativa): Contradiz H₀
    ↓
Escolher Nível de Significância α
  ├─ Comum: 0.05, 0.01, 0.001
  └─ Pequeno α → Teste mais rigoroso
    ↓
Estatística de Teste
  ├─ Parametrizado (z, t, F, χ²)
  └─ Não-parametrizado (rankMann-Whitney, Wilcoxon)
    ↓
Cálculo do p-value
  ├─ P(observado | H₀)
  └─ Pequeno p → Rejeita H₀
    ↓
Decisão
  ├─ p < α: Rejeitar H₀
  └─ p ≥ α: Falhar em rejeitar H₀
```

### Tipos de Erro em Testes de Hipóteses

| | H₀ Verdadeira | H₀ Falsa |
|---|---|---|
| **Rejeitar H₀** | Erro Tipo I (α) | Correto |
| **Não Rejeitar H₀** | Correto | Erro Tipo II (β) |

Potência = 1 - β

### Correlação e Regressão

```
Associação entre Variáveis
    ↓
Correlação (associação linear)
  ├─ Pearson r: variáveis contínuas
  ├─ Spearman ρ: postos/ranques
  └─ -1 ≤ coeff ≤ 1
    ↓
Regressão (predição)
  ├─ Linear: y = a + bx
  ├─ Múltipla: y = b₀ + b₁x₁ + ... + bₚxₚ
  └─ Não-linear: polinomial, exponencial
    ↓
Qualidade do Ajuste
  ├─ R² (coeff. determinação): 0 a 1
  ├─ RMSE (erro médio quadrático)
  └─ Resíduos (y - ŷ)
```

### Métodos Computacionais Avançados

1. **Boostrap**: Reamostragem para intervalos de confiança
2. **Simulação de Monte Carlo**: Aproximar distribuições complexas
3. **MCMC**: Amostragem de distribuições posteriores
4. **Máxima Verossimilhança**: Estimação de parâmetros